# 2장 — Attention에서 Transformer와 GPT까지

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch02_transformer_gpt.ipynb)

이 노트북은 『밑바닥부터 시작하는 딥러닝 6』의 공식 코드 저장소를 기준으로 구성했습니다. T4에서 실행하기 어렵다는 이유로 알고리즘이나 모델 구조를 토이 버전으로 바꾸지 않습니다.

- 기준 upstream commit: `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`
- 포함한 장 코드 파일 수: **9개**
- 함께 펼쳐서 보여주는 공통 모듈 수: **1개**


## 노트북 구성 원칙

1. 공식 `.py`의 모델 구조와 계산 로직을 그대로 유지합니다.
2. 함수·클래스·실행부를 셀 단위로 나눠 위에서 아래로 읽기 쉽게 배치합니다.
3. 일본어 자연어 주석은 한국어로 바꾸며, 변수명·수식·텐서 shape 같은 기술 표기는 유지합니다.
4. 공통 `codebot` / `storybot` 모듈도 외부 파일 뒤에 숨기지 않고 이 노트북에서 직접 확인할 수 있게 합니다.
5. T4에서 시간이 오래 걸리는 전체 학습 스케줄도 기본값 자체를 임의 축소하지 않습니다.


## 0. Colab 환경 준비

먼저 공식 저장소를 고정된 커밋으로 준비하고 현재 런타임의 GPU를 확인합니다.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('작업 경로:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 확인 중 오류:', exc)


## 1. 이 장에서 사용하는 공통 구현

장 코드가 import하는 로컬 모듈을 먼저 읽습니다. 긴 파일도 클래스·함수 단위로 나눠 표시합니다.


### `codebot/model.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile codebot/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


#### `MultiHeadAttention` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, dropout_rate=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)  # 출력 예시: (B, C, H*D)
        K = self.W_k(x)  # 출력 예시: (B, C, H*D)
        V = self.W_v(x)  # 출력 예시: (B, C, H*D)

        Q = Q.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)
        K = K.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)
        V = V.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)

        scores = torch.matmul(Q, K.transpose(-2, -1))  # 출력 예시: (B, H, C, C)
        scores = scores / (D ** 0.5)

        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)  # 출력 예시: (B, H, C, C)
        weights = self.attention_dropout(weights)
        hidden = torch.matmul(weights, V)  # 출력 예시: (B, H, C, D)

        hidden = hidden.transpose(1, 2).contiguous()  # 출력 예시: (B, C, H, D)
        hidden = hidden.view(B, C, H * D)  # 출력 예시: (B, C, H*D)
        output = self.W_o(hidden)  # 출력 예시: (B, C, E)
        output = self.output_dropout(output)

        return output


#### `LayerNorm` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta


#### `GELU` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


#### `FFN` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class FFN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),  # 참고: GELU()
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.layers(x)


#### `Block` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.LayerNorm(embed_dim)  # 참고: LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = nn.LayerNorm(embed_dim)  # 참고: LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


#### `GPT` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, dropout_rate):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_context_len, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, dropout_rate)
            for _ in range(n_layer)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size)

        self.embed.weight = self.unembed.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, C = ids.shape
        device = ids.device

        pos = torch.arange(0, C, dtype=torch.long, device=device)
        emb = self.embed(ids)
        pos_emb = self.pos_embed(pos)
        x = self.dropout(emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        logits = self.unembed(x)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            dropout_rate=checkpoint['dropout_rate']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model


## 2. 장별 실습 코드

공식 저장소의 장 코드를 파일 순서대로 모두 다룹니다.


## `ch02/01_soft_dict.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import numpy as np


### 설정 및 값 준비: `d`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
d = {
    'apple': 100,
    'banana': 200,
    'cherry': 300,
    'durian': 400
}


### 설정 및 값 준비: `query`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
query = 'banana'


### 실행 및 결과 확인


In [ ]:
print(d[query])  # 참고: 200


### 설정 및 값 준비: `movie_preferences`


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
# 이 코드 단계의 동작을 확인하는 예시
movie_preferences = {
    (8, 2, 3): 85,  # 이 코드 단계의 동작을 확인하는 예시
    (3, 9, 1): 70,  # 이 코드 단계의 동작을 확인하는 예시
    (1, 2, 9): 60,  # 이 코드 단계의 동작을 확인하는 예시
    (5, 5, 5): 75,  # 이 코드 단계의 동작을 확인하는 예시
    (7, 6, 2): 80,  # 이 코드 단계의 동작을 확인하는 예시
    (2, 7, 6): 65,  # 이 코드 단계의 동작을 확인하는 예시
    (9, 1, 1): 90,  # 이 코드 단계의 동작을 확인하는 예시
}


### 설정 및 값 준비: `new_movie`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
new_movie = (6, 4, 5)


### `soft_dictionary()` 함수 구현


In [ ]:

def soft_dictionary(query, dictionary):
    # 이 코드 단계의 동작을 확인하는 예시
    similarity = []
    for key in dictionary:
        s = np.dot(query, key)
        similarity.append(s)

    # 이 코드 단계의 동작을 확인하는 예시
    exp_similarity = np.exp(similarity)
    weights = exp_similarity / np.sum(exp_similarity)

    # 이 코드 단계의 동작을 확인하는 예시
    result = 0
    for weight, value in zip(weights, dictionary.values()):
        result += weight * value

    return result, weights


### 실행 코드


In [ ]:


predicted_rating, weights = soft_dictionary(new_movie, movie_preferences)


### 실행 및 결과 확인


In [ ]:

print(f"新しい映画 {new_movie} の予測評価: {predicted_rating:.2f} 点")


### 실행 및 결과 확인


In [ ]:
print("\n各映画の重み:")


### 반복 실행


In [ ]:
for key, weight in zip(movie_preferences.keys(), weights):
    print(f"映画 {key}: {weight*100:.2f}%")


## `ch02/02_attn_math.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn.functional as F


### 설정 및 값 준비: `K`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
K = torch.tensor([
    [8, 2, 3],  # 이 코드 단계의 동작을 확인하는 예시
    [3, 9, 1],  # 이 코드 단계의 동작을 확인하는 예시
    [1, 2, 9],  # 이 코드 단계의 동작을 확인하는 예시
    [5, 5, 5],  # 이 코드 단계의 동작을 확인하는 예시
    [7, 6, 2],  # 이 코드 단계의 동작을 확인하는 예시
    [2, 7, 6],  # 이 코드 단계의 동작을 확인하는 예시
    [9, 1, 1],  # 이 코드 단계의 동작을 확인하는 예시
], dtype=torch.float32)


### 설정 및 값 준비: `V`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
V = torch.tensor([
    [85],
    [70],
    [60],
    [75],
    [80],
    [65],
    [90]
], dtype=torch.float32)


### 설정 및 값 준비: `Q`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
Q = torch.tensor([
    [6, 4, 5],  # 이 코드 단계의 동작을 확인하는 예시
    [2, 8, 3],  # 이 코드 단계의 동작을 확인하는 예시
    [4, 3, 7],  # 이 코드 단계의 동작을 확인하는 예시
], dtype=torch.float32)


### `attention()` 함수 구현


In [ ]:

def attention(Q, K, V):
    similarity = torch.matmul(Q, K.t())     # 이 코드 단계의 동작을 확인하는 예시
    weights = F.softmax(similarity, dim=1)  # 이 코드 단계의 동작을 확인하는 예시
    output = torch.matmul(weights, V)       # 이 코드 단계의 동작을 확인하는 예시
    return output, weights


### 실행 코드


In [ ]:

predicted_ratings, weights = attention(Q, K, V)


### 반복 실행


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
for movie, rating in zip(Q, predicted_ratings):
    print(f"映画 {movie.numpy()} の予測評価: {rating.item():.2f}")


## `ch02/03_attn_scaling.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn.functional as F


### 설정 및 값 준비: `x`


In [ ]:

x = torch.tensor([100.0, 200.0, 300.0])


### 설정 및 값 준비: `y`


In [ ]:
y = F.softmax(x, dim=0)


### 실행 및 결과 확인


In [ ]:
print(y)


### 필요한 라이브러리와 모듈 불러오기


In [ ]:


import numpy as np
import matplotlib.pyplot as plt


### 설정 및 값 준비: `d`


In [ ]:

d = 10


### 설정 및 값 준비: `q`


In [ ]:

q = np.random.randn(d)


### 설정 및 값 준비: `k`


In [ ]:
k = np.random.randn(d)


### 설정 및 값 준비: `dot_product`


In [ ]:

dot_product = np.dot(q, k)


### 설정 및 값 준비: `scaled_dot_product`


In [ ]:
scaled_dot_product = dot_product / np.sqrt(d)


### 실행 및 결과 확인


In [ ]:

print('dot product:', dot_product)


### 실행 및 결과 확인


In [ ]:
print('scaled dot product:', scaled_dot_product)


### 설정 및 값 준비: `d`


In [ ]:


d = 10


### 설정 및 값 준비: `num_samples`


In [ ]:
num_samples = 10000  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `dot_products`


In [ ]:

dot_products = []


### 설정 및 값 준비: `scaled_dot_products`


In [ ]:
scaled_dot_products = []


### 반복 실행


In [ ]:

for _ in range(num_samples):
    q = np.random.randn(d)
    k = np.random.randn(d)

    dot_product = np.dot(q, k)
    scaled_dot_product = dot_product / np.sqrt(d)

    dot_products.append(dot_product)
    scaled_dot_products.append(scaled_dot_product)


### 실행 및 결과 확인


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
plt.figure(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:
plt.hist(dot_products, bins=50, alpha=0.5, label='Without scaling')


### 실행 및 결과 확인


In [ ]:
plt.hist(scaled_dot_products, bins=50, alpha=0.5, label='With scaling')


### 실행 및 결과 확인


In [ ]:
plt.legend()


### 실행 및 결과 확인


In [ ]:
plt.show()


### 실행 및 결과 확인


In [ ]:

print("Variances without scaling:", np.var(dot_products))


### 실행 및 결과 확인


In [ ]:
print("Variances with scaling:", np.var(scaled_dot_products))


## `ch02/06_attn_mask.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### `Attention` 클래스 구현


In [ ]:

class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        # 이 코드 단계의 동작을 확인하는 예시
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

        self.key_dim = key_dim

    def forward(self, x):  # 텐서 크기: x: (B, C, E)
        Q = self.W_q(x)    # 텐서 크기: Q: (B, C, D)
        K = self.W_k(x)    # 텐서 크기: K: (B, C, D)
        V = self.W_v(x)    # 텐서 크기: V: (B, C, E)

        # 이 코드 단계의 동작을 확인하는 예시
        K_t = K.transpose(-2, -1)  # 출력 예시: (B, D, C)
        scores = torch.matmul(Q, K_t)  # 출력 예시: (B, C, C)
        scores = scores / (self.key_dim ** 0.5)

        # 마스크의적용
        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)  # 출력 예시: (B, C, E)
        return output


### 설정 및 값 준비: `attention`


In [ ]:

attention = Attention(embed_dim=256, key_dim=64)


### 설정 및 값 준비: `x`


In [ ]:
x = torch.randn(2, 5, 256)  # 출력 예시: (batch_size=2, context_len=5, embed_dim=256)


### 설정 및 값 준비: `y`


In [ ]:
y = attention(x)


### 실행 및 결과 확인


In [ ]:

print("入力形状:", x.shape)


### 실행 및 결과 확인


In [ ]:
print("出力形状:", y.shape)


## `ch02/07_attn_value.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### `Attention` 클래스 구현


In [ ]:

class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_o = nn.Linear(key_dim, embed_dim, bias=False)  # 이 코드 단계의 동작을 확인하는 예시
        self.key_dim = key_dim

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        K_t = K.transpose(-2, -1)
        scores = torch.matmul(Q, K_t)
        scores = scores / (self.key_dim ** 0.5)

        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        # 출력변환
        output = self.W_o(hidden)

        return output


### 설정 및 값 준비: `attention`


In [ ]:

attention = Attention(embed_dim=256, key_dim=64)


### 설정 및 값 준비: `x`


In [ ]:
x = torch.randn(2, 5, 256)


### 설정 및 값 준비: `y`


In [ ]:
y = attention(x)


### 실행 및 결과 확인


In [ ]:

print("入力形状:", x.shape)


### 실행 및 결과 확인


In [ ]:
print("出力形状:", y.shape)


## `ch02/08_multi_head.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### 설정 및 값 준비: `B`


In [ ]:

B = 2   # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `C`


In [ ]:
C = 4   # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `E`


In [ ]:
E = 16  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `H`


In [ ]:
H = 3   # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `D`


In [ ]:
D = 8   # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `x`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
x = torch.randn(B, C, E)


### 설정 및 값 준비: `W_q`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
W_q = nn.Linear(E, H*D, bias=False)


### 설정 및 값 준비: `W_k`


In [ ]:
W_k = nn.Linear(E, H*D, bias=False)


### 설정 및 값 준비: `W_v`


In [ ]:
W_v = nn.Linear(E, H*D, bias=False)


### 설정 및 값 준비: `Q`


In [ ]:

Q = W_q(x)  # 출력 예시: (B, C, H*D)


### 설정 및 값 준비: `K`


In [ ]:
K = W_k(x)  # 출력 예시: (B, C, H*D)


### 설정 및 값 준비: `V`


In [ ]:
V = W_v(x)  # 출력 예시: (B, C, H*D)


### 설정 및 값 준비: `Q`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
Q = Q.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)


### 설정 및 값 준비: `K`


In [ ]:
K = K.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)


### 설정 및 값 준비: `V`


In [ ]:
V = V.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)


### 설정 및 값 준비: `scores`


In [ ]:

scores = torch.matmul(Q, K.transpose(-2, -1))  # 출력 예시: (B, H, C, C)


### 설정 및 값 준비: `scores`


In [ ]:
scores = scores / (D ** 0.5)


### 설정 및 값 준비: `mask`


In [ ]:

# 마스크처리
mask = torch.tril(torch.ones(C, C, device=scores.device))


### 설정 및 값 준비: `scores`


In [ ]:
scores = scores.masked_fill(mask == 0, float('-inf'))


### 설정 및 값 준비: `weights`


In [ ]:

# Attention가중치
weights = F.softmax(scores, dim=-1)  # 출력 예시: (B, H, C, C)


### 설정 및 값 준비: `hidden`


In [ ]:
hidden = torch.matmul(weights, V)   # 출력 예시: (B, H, C, D)


### 설정 및 값 준비: `hidden`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
hidden = hidden.transpose(1, 2)  # 출력 예시: (B, C, H, D)


### 설정 및 값 준비: `hidden`


In [ ]:
hidden = hidden.contiguous().view(B, C, H*D)  # 출력 예시: (B, C, H*D)


### 설정 및 값 준비: `W_o`


In [ ]:

# 출력변환: (B, C, H*D) → (B, C, E)
W_o = nn.Linear(H*D, E, bias=False)


### 설정 및 값 준비: `output`


In [ ]:
output = W_o(hidden)  # 출력 예시: (B, C, E)


### `MultiHeadAttention` 클래스 구현


In [ ]:


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, dropout_rate=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        # 이 코드 단계의 동작을 확인하는 예시
        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, C, E = x.shape  # 이 코드 단계의 동작을 확인하는 예시
        H, D = self.n_head, self.head_dim  # 이 코드 단계의 동작을 확인하는 예시

        # Q, K, V 의계산
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # 이 코드 단계의 동작을 확인하는 예시
        Q = Q.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)
        K = K.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)
        V = V.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)

        scores = torch.matmul(Q, K.transpose(-2, -1))  # 출력 예시: (B, H, C, C)
        scores = scores / (D ** 0.5)

        # 마스크처리
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # Attention가중치
        weights = F.softmax(scores, dim=-1)  # 출력 예시: (B, H, C, C)
        weights = self.attention_dropout(weights)  # 이 코드 단계의 동작을 확인하는 예시
        hidden = torch.matmul(weights, V)  # 출력 예시: (B, H, C, D)

        # 이 코드 단계의 동작을 확인하는 예시
        hidden = hidden.transpose(1, 2).contiguous()  # 출력 예시: (B, C, H, D)
        hidden = hidden.view(B, C, H * D)  # 출력 예시: (B, C, H*D)
        output = self.W_o(hidden)  # 출력 예시: (B, C, E)
        output = self.output_dropout(output)  # 이 코드 단계의 동작을 확인하는 예시

        return output


### 설정 및 값 준비: `embed_dim`


In [ ]:

# 사용 예시
embed_dim = 512


### 설정 및 값 준비: `n_head`


In [ ]:
n_head = 8


### 설정 및 값 준비: `head_dim`


In [ ]:
head_dim = 64


### 설정 및 값 준비: `mha`


In [ ]:

mha = MultiHeadAttention(embed_dim, n_head, head_dim)


### 설정 및 값 준비: `batch_size`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
batch_size = 2


### 설정 및 값 준비: `context_len`


In [ ]:
context_len = 10


### 설정 및 값 준비: `x`


In [ ]:
x = torch.randn(batch_size, context_len, embed_dim)


### 설정 및 값 준비: `output`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
output = mha(x)


### 실행 및 결과 확인


In [ ]:
print(f"入力形状: {x.shape}")       # 출력 예시: (2, 10, 512)


### 실행 및 결과 확인


In [ ]:
print(f"出力形状: {output.shape}") # 출력 예시: (2, 10, 512)


## `ch02/09_norm_gelu.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
from codebot.model import MultiHeadAttention


### `LayerNorm` 클래스 구현


In [ ]:


class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta


### `GELU` 클래스 구현


In [ ]:


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


### `FFN` 클래스 구현


In [ ]:

class FFN(nn.Module):
    def __init__(self, x_dim, hidden_dim=None, dropout_rate=0.1):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(4 * x_dim)

        self.layers = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            GELU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.layers(x)


### `Block` 클래스 구현


In [ ]:


class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim=None, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


## `ch02/10_gpt2.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import torch
import torch.nn as nn
from codebot.model import Block


### `GPT` 클래스 구현


In [ ]:


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, dropout_rate):
        super().__init__()
        self.vocab_size = vocab_size            # 어휘 크기
        self.max_context_len = max_context_len  # 이 코드 단계의 동작을 확인하는 예시
        self.embed_dim = embed_dim              # 이 코드 단계의 동작을 확인하는 예시
        self.n_head = n_head                    # 이 코드 단계의 동작을 확인하는 예시
        self.n_layer = n_layer                  # 이 코드 단계의 동작을 확인하는 예시
        self.ff_dim = ff_dim                    # 이 코드 단계의 동작을 확인하는 예시
        self.dropout_rate = dropout_rate        # 이 코드 단계의 동작을 확인하는 예시

        # 임베딩층
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_context_len, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

        # 이 코드 단계의 동작을 확인하는 예시
        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, dropout_rate)
            for _ in range(n_layer)
        ])

        # 출력층
        self.norm = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size)

        # 이 코드 단계의 동작을 확인하는 예시
        self.embed.weight = self.unembed.weight

        # 가중치의초기화
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, C = ids.shape  # 이 코드 단계의 동작을 확인하는 예시
        device = ids.device

        # 임베딩
        pos = torch.arange(0, C, dtype=torch.long, device=device)
        emb = self.embed(ids)
        pos_emb = self.pos_embed(pos)
        x = self.dropout(emb + pos_emb)

        # 이 코드 단계의 동작을 확인하는 예시
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        # 출력
        logits = self.unembed(x)  # 출력 예시: (B, C, vocab_size)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            dropout_rate=checkpoint['dropout_rate']
        )
        # 가중치의불러오기
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model


### 설정 및 값 준비: `vocab_size`


In [ ]:


vocab_size = 1000


### 설정 및 값 준비: `max_context_len`


In [ ]:
max_context_len = 256


### 설정 및 값 준비: `embed_dim`


In [ ]:
embed_dim = 384


### 설정 및 값 준비: `n_head`


In [ ]:
n_head = 6


### 설정 및 값 준비: `n_layer`


In [ ]:
n_layer = 6


### 설정 및 값 준비: `ff_dim`


In [ ]:
ff_dim = 4 * embed_dim


### 설정 및 값 준비: `dropout_rate`


In [ ]:
dropout_rate = 0.1


### 설정 및 값 준비: `model`


In [ ]:

# 모델생성
model = GPT(vocab_size, max_context_len, embed_dim, n_head,
             n_layer, ff_dim, dropout_rate)


### 설정 및 값 준비: `dummy_input`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
dummy_input = torch.randint(0, vocab_size, (1, max_context_len))


### 설정 및 값 준비: `logits`


In [ ]:
logits = model(dummy_input)


### 실행 및 결과 확인


In [ ]:
print(f"出力形状: {logits.shape}")


## `ch02/graph.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


### 설정 및 값 준비: `x`


In [ ]:

# 데이터 생성
x = np.linspace(-3, 3, 500)


### 설정 및 값 준비: `relu`


In [ ]:

# ReLU함수
relu = np.maximum(0, x)


### 설정 및 값 준비: `gelu`


In [ ]:

# GELU함수（근사식）
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


### 실행 코드


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
fig, ax = plt.subplots(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:

ax.plot(x, relu, 'b-', linewidth=2, label='ReLU')


### 실행 및 결과 확인


In [ ]:
ax.plot(x, gelu, 'r-', linewidth=2, label='GELU')


### 실행 및 결과 확인


In [ ]:

# 축 설정
ax.set_xlim(-3, 3)


### 실행 및 결과 확인


In [ ]:
ax.set_ylim(-0.5, 3.0)


### 실행 및 결과 확인


In [ ]:
ax.set_xlabel('x', fontsize=12)


### 실행 및 결과 확인


In [ ]:
ax.set_ylabel('f(x)', fontsize=12)


### 실행 및 결과 확인


In [ ]:

# 격자
ax.grid(True, linestyle='--', alpha=0.7)


### 실행 및 결과 확인


In [ ]:
ax.axhline(y=0, color='gray', linewidth=0.5)


### 실행 및 결과 확인


In [ ]:
ax.axvline(x=0, color='gray', linewidth=0.5)


### 실행 및 결과 확인


In [ ]:

# 범례
ax.legend(loc='upper left', fontsize=12)


### 실행 및 결과 확인


In [ ]:

# 여백 조정
plt.tight_layout()


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
plt.savefig('relu_gelu.png', format='png', bbox_inches='tight')


### 실행 및 결과 확인


In [ ]:
plt.close()


## T4 실행 메모

위 코드는 공식 구현의 모델 구조·알고리즘·기본 하이퍼파라미터를 보존합니다. 학습 시간이 긴 셀은 T4에서도 실행 자체는 가능할 수 있지만 전체 스텝 완주에는 시간이 많이 필요할 수 있습니다. 이 노트북은 빠른 실행을 위해 모델을 임의로 축소하거나 핵심 계산을 생략하지 않습니다.
